# 01 — Data Exploration
## Credit Risk: Loss Given Default (LGD) Prediction

**Objectives:**
- Understand the Freddie Mac SFLP dataset structure
- Examine LGD target distribution (spikes at 0 and 1)
- Identify key feature distributions and correlations
- Quantify missingness and assess imputation strategy
- Validate data volume by vintage year (2010–2015)

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config import load_config

config = load_config()

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Config loaded. Data dirs:', config.data.raw_dir)

## 1. Load Interim Data

Load the staged interim files from `data/interim/`. If they don't exist yet, run `python src/data/ingest.py` and `python src/data/preprocess.py` first.

In [ ]:
interim_dir = Path(config.data.interim_dir)
processed_dir = Path(config.data.processed_dir)

# Load cleaned merged dataset (post-preprocessing)
cleaned_path = processed_dir / 'cleaned.parquet'
if cleaned_path.exists():
    df = pd.read_parquet(cleaned_path)
    print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
    print(df.dtypes)
else:
    print('Cleaned data not found. Run the data pipeline first.')
    print('  python src/data/ingest.py')
    print('  python src/data/preprocess.py')

## 2. LGD Target Distribution

LGD has a characteristic bi-modal distribution: spikes at 0 (full recovery) and 1 (total loss), with a roughly uniform middle. This motivates the tail-weighted loss function in LGDNet.

In [ ]:
# TODO: Plot LGD distribution
# Expected: bi-modal spikes at 0 and 1, roughly uniform middle

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['loss_given_default'], bins=50, edgecolor='white', color='#2563EB', alpha=0.8)
axes[0].set_xlabel('LGD')
axes[0].set_ylabel('Count')
axes[0].set_title('LGD Distribution')
axes[0].axvline(df['loss_given_default'].mean(), color='red', linestyle='--', label=f'Mean: {df["loss_given_default"].mean():.3f}')
axes[0].legend()

# CDF
sorted_lgd = np.sort(df['loss_given_default'])
cdf = np.arange(1, len(sorted_lgd) + 1) / len(sorted_lgd)
axes[1].plot(sorted_lgd, cdf, color='#2563EB')
axes[1].set_xlabel('LGD')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('LGD Empirical CDF')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('LGD summary statistics:')
print(df['loss_given_default'].describe())
print(f"\nLGD = 0.0: {(df['loss_given_default'] == 0).sum():,} ({(df['loss_given_default'] == 0).mean()*100:.1f}%)")
print(f"LGD = 1.0: {(df['loss_given_default'] == 1).sum():,} ({(df['loss_given_default'] == 1).mean()*100:.1f}%)")

## 3. Vintage Year Distribution

In [ ]:
# TODO: Default count and mean LGD by vintage year
# Expected: 2010-2012 vintages have more defaults (post-crisis peak)

if 'vintage_year' in df.columns:
    vintage_stats = df.groupby('vintage_year').agg(
        count=('loss_given_default', 'count'),
        mean_lgd=('loss_given_default', 'mean'),
        median_lgd=('loss_given_default', 'median')
    ).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(vintage_stats['vintage_year'], vintage_stats['count'], color='#2563EB', alpha=0.8)
    axes[0].set_xlabel('Vintage Year')
    axes[0].set_ylabel('Number of Defaulted Loans')
    axes[0].set_title('Default Count by Vintage Year')
    
    axes[1].plot(vintage_stats['vintage_year'], vintage_stats['mean_lgd'], 'o-', color='#2563EB', label='Mean LGD')
    axes[1].plot(vintage_stats['vintage_year'], vintage_stats['median_lgd'], 's--', color='#DC2626', label='Median LGD')
    axes[1].set_xlabel('Vintage Year')
    axes[1].set_ylabel('LGD')
    axes[1].set_title('Mean/Median LGD by Vintage Year')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print(vintage_stats)

## 4. Feature Distributions

In [ ]:
# TODO: Distributions of key numeric features
numeric_features = ['orig_ltv', 'orig_upb', 'orig_interest_rate', 'orig_term']
numeric_available = [f for f in numeric_features if f in df.columns]

if numeric_available:
    df[numeric_available].hist(bins=30, figsize=(14, 10), color='#2563EB', alpha=0.7, edgecolor='white')
    plt.suptitle('Origination Feature Distributions', y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Categorical Feature Breakdown

In [ ]:
# TODO: Mean LGD by property type, occupancy status, channel
# These are the segment columns for the benchmark model

categorical_features = ['property_type', 'occupancy_status', 'channel']

for col in categorical_features:
    if col in df.columns:
        segment_stats = df.groupby(col)['loss_given_default'].agg(['mean', 'count']).reset_index()
        segment_stats = segment_stats.sort_values('mean', ascending=False)
        
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.barh(segment_stats[col].astype(str), segment_stats['mean'], color='#2563EB', alpha=0.8)
        ax.set_xlabel('Mean LGD')
        ax.set_title(f'Mean LGD by {col}')
        plt.tight_layout()
        plt.show()
        print(segment_stats.to_string(index=False))
        print()

## 6. Missingness Analysis

In [ ]:
# TODO: Null rates per column
null_rates = df.isna().mean().sort_values(ascending=False)
null_rates_nonzero = null_rates[null_rates > 0]

if len(null_rates_nonzero) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(null_rates_nonzero) * 0.3)))
    null_rates_nonzero.plot(kind='barh', ax=ax, color='#DC2626', alpha=0.7)
    ax.set_xlabel('Null Rate')
    ax.set_title('Missingness by Feature')
    ax.axvline(0.05, color='black', linestyle='--', alpha=0.5, label='5% threshold')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(null_rates_nonzero)
else:
    print('No missing values in the cleaned dataset.')

## 7. Correlation Analysis

In [ ]:
# TODO: Correlation of numeric features with LGD target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'loss_given_default' in numeric_cols:
    corr_with_target = df[numeric_cols].corr()['loss_given_default'].drop('loss_given_default').sort_values()
    
    fig, ax = plt.subplots(figsize=(8, max(5, len(corr_with_target) * 0.3)))
    colors = ['#DC2626' if x < 0 else '#2563EB' for x in corr_with_target]
    corr_with_target.plot(kind='barh', ax=ax, color=colors, alpha=0.8)
    ax.set_xlabel('Pearson Correlation with LGD')
    ax.set_title('Feature Correlations with LGD Target')
    ax.axvline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()

## Key Findings

*To be completed after analysis:*

1. **LGD distribution:** [Describe the bi-modal pattern observed]
2. **Vintage trends:** [Describe how LGD varies by origination year]
3. **Top predictors:** [List features most correlated with LGD]
4. **Missingness:** [Key fields with missing values and chosen strategy]
5. **Segment differences:** [Notable LGD differences by property type / occupancy]